# Anime Recommender · Notebook 2: 訓練三個傳統推薦模型

三個模型 + 一個 baseline:
1. **Popularity** (baseline) — 在 03_evaluate 才會計算,這裡不用訓練
2. **Content-Based** — Genre 用 multi-hot、Type 用 one-hot,cosine similarity
3. **User-Based CF** — KNN on sparse user-item matrix
4. **SVD (Matrix Factorization)** — scipy 的 sparse SVD,把 user_factors / item_factors / biases 拆出來儲存

> 設計重點:SVD 模型不依賴 `scikit-surprise`,而是輸出純 numpy 陣列,本地端只需 numpy 就能 inference,
> 在 Windows 上不會出現套件安裝問題。

## Step 1 — 掛載 Drive 並讀取上一步的資料

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pickle
import pandas as pd
import numpy as np
from scipy import sparse

ARTIFACTS_DIR = '/content/drive/MyDrive/anime-recsys/artifacts'

anime_meta = pd.read_parquet(f'{ARTIFACTS_DIR}/anime_meta.parquet')
train_df = pd.read_parquet(f'{ARTIFACTS_DIR}/train.parquet')
with open(f'{ARTIFACTS_DIR}/mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)

user_id_to_idx = mappings['user_id_to_idx']
anime_id_to_idx = mappings['anime_id_to_idx']
N_USERS = len(user_id_to_idx); N_ITEMS = len(anime_id_to_idx)
print(f'#users = {N_USERS:,}, #items = {N_ITEMS:,}, train ratings = {len(train_df):,}')

## Step 2 — 建立 train 的 sparse user-item matrix (給 User-CF 與 SVD 共用)

In [ ]:
rows = train_df['user_id'].map(user_id_to_idx).to_numpy()
cols = train_df['anime_id'].map(anime_id_to_idx).to_numpy()
data = train_df['rating'].to_numpy(dtype=np.float32)

user_item = sparse.csr_matrix((data, (rows, cols)), shape=(N_USERS, N_ITEMS))
print(f'user_item shape = {user_item.shape}, nnz = {user_item.nnz:,}, density = {user_item.nnz/(N_USERS*N_ITEMS):.5%}')

## Step 3 — Model A:Content-Based

- **特徵**:Genre 做 multi-hot、Type 做 one-hot
- **相似度**:cosine similarity 計出 item × item 相似度矩陣
- **稀疏化**:只保留每部作品 top-50 的鄰居,大幅縮小檔案

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize

# 整理出依 idx 排序的 anime metadata
meta = anime_meta.set_index('anime_id').loc[list(anime_id_to_idx.keys())].reset_index()
meta['genre'] = meta['genre'].fillna('')
meta['type'] = meta['type'].fillna('Unknown')

# Genre multi-hot (CountVectorizer with binary=True)
cv_genre = CountVectorizer(tokenizer=lambda s: [t.strip() for t in s.split(',') if t.strip()],
                           lowercase=False, binary=True, token_pattern=None)
X_genre = cv_genre.fit_transform(meta['genre'])
print(f'#genres = {X_genre.shape[1]}')

# Type one-hot
cv_type = CountVectorizer(tokenizer=lambda s: [s], lowercase=False, token_pattern=None)
X_type = cv_type.fit_transform(meta['type'])
print(f'#types = {X_type.shape[1]}')

# 合併特徵 + L2 normalize 後算 cosine 等同點積
X = sparse.hstack([X_genre, X_type]).tocsr().astype(np.float32)
X = normalize(X, norm='l2', axis=1)
print(f'X shape = {X.shape}')

In [ ]:
# 計算 item-item 相似度。為了讓檔案不要太大,只保留每行 top-K
from sklearn.neighbors import NearestNeighbors

TOP_K_ITEM = 50
nn = NearestNeighbors(n_neighbors=TOP_K_ITEM + 1, metric='cosine', n_jobs=-1)
nn.fit(X)
dists, idxs = nn.kneighbors(X)  # 包含自己

# 轉成稀疏矩陣 (item × item,相似度 = 1 - distance)
sim_rows = np.repeat(np.arange(N_ITEMS), TOP_K_ITEM)
sim_cols = idxs[:, 1:].ravel()                # 跳過第一個 (自己)
sim_vals = (1 - dists[:, 1:]).ravel().astype(np.float32)
content_sim = sparse.csr_matrix((sim_vals, (sim_rows, sim_cols)), shape=(N_ITEMS, N_ITEMS))
print(f'content_sim nnz = {content_sim.nnz:,}')

sparse.save_npz(f'{ARTIFACTS_DIR}/content_sim.npz', content_sim)
print(f'✅ Saved content_sim.npz')

## Step 4 — Model B:User-Based Collaborative Filtering

- 對每個 user 預先找 K=30 個最相似的 user (cosine on rating vectors)
- 推薦時把 K 位鄰居的評分用相似度加權平均
- 本地端 inference 時只要這個 KNN 結構 + user-item matrix 即可

In [ ]:
from sklearn.neighbors import NearestNeighbors

K_NEIGHBORS = 30
user_item_norm = normalize(user_item, norm='l2', axis=1)
nn_user = NearestNeighbors(n_neighbors=K_NEIGHBORS + 1, metric='cosine', n_jobs=-1)
nn_user.fit(user_item_norm)
u_dists, u_idxs = nn_user.kneighbors(user_item_norm)

neighbors_idx = u_idxs[:, 1:].astype(np.int32)              # (N_USERS, K)
neighbors_sim = (1 - u_dists[:, 1:]).astype(np.float32)     # (N_USERS, K)
print(f'neighbors_idx shape = {neighbors_idx.shape}')

In [ ]:
user_cf_bundle = {
    'user_item': user_item,
    'neighbors_idx': neighbors_idx,
    'neighbors_sim': neighbors_sim,
}
with open(f'{ARTIFACTS_DIR}/user_cf_model.pkl', 'wb') as f:
    pickle.dump(user_cf_bundle, f)
print('✅ Saved user_cf_model.pkl')

## Step 5 — Model C:SVD (Matrix Factorization)

### 數學步驟
1. 算 `global_mean`、`user_bias`、`item_bias`
2. 把評分矩陣中心化:`R_centered = R - global - user_bias - item_bias`
3. 對中心化後的稀疏矩陣做 truncated SVD:`R_centered ≈ U·Σ·V^T`
4. `user_factors = U·sqrt(Σ)`、`item_factors = V·sqrt(Σ)`
5. 預測:`r̂_{ui} = global + b_u + b_i + user_factors[u] · item_factors[i]`

In [ ]:
N_FACTORS = 50

# 1. global mean
global_mean = float(user_item.data.mean())
print(f'global mean = {global_mean:.3f}')

# 2. user mean & item mean (只算 nnz 部分)
user_sum = np.asarray(user_item.sum(axis=1)).ravel()
user_cnt = np.diff(user_item.indptr)               # 每個 user 的 nnz 數
user_mean = np.where(user_cnt > 0, user_sum / np.maximum(user_cnt, 1), global_mean)
user_bias = (user_mean - global_mean).astype(np.float32)

item_sum = np.asarray(user_item.sum(axis=0)).ravel()
item_cnt = np.asarray((user_item != 0).sum(axis=0)).ravel()
item_mean = np.where(item_cnt > 0, item_sum / np.maximum(item_cnt, 1), global_mean)
item_bias = (item_mean - global_mean).astype(np.float32)
print(f'user_bias range = [{user_bias.min():.2f}, {user_bias.max():.2f}]')
print(f'item_bias range = [{item_bias.min():.2f}, {item_bias.max():.2f}]')

In [ ]:
# 3. 把 R 中心化 (只對 observed 評分做減法,保持稀疏性)
R = user_item.copy().astype(np.float32)
rows_idx, cols_idx = R.nonzero()
R.data = (R.data - global_mean - user_bias[rows_idx] - item_bias[cols_idx]).astype(np.float32)
print(f'R_centered nnz = {R.nnz:,}, mean = {R.data.mean():.4f}')

In [ ]:
# 4. Truncated SVD on sparse centered matrix
from scipy.sparse.linalg import svds

U, S, Vt = svds(R, k=N_FACTORS)
# svds 回傳的奇異值是升序,翻轉
order = np.argsort(-S)
S = S[order]; U = U[:, order]; Vt = Vt[order, :]

sqrt_S = np.sqrt(S).astype(np.float32)
user_factors = (U * sqrt_S).astype(np.float32)
item_factors = (Vt.T * sqrt_S).astype(np.float32)
print(f'user_factors shape = {user_factors.shape}, item_factors shape = {item_factors.shape}')
print(f'top singular values: {S[:5]}')

In [ ]:
# 5. 存 SVD bundle (純 numpy,本地端不需要 surprise 套件)
svd_bundle = {
    'global_mean': float(global_mean),
    'user_bias': user_bias,
    'item_bias': item_bias,
    'user_factors': user_factors,
    'item_factors': item_factors,
    'n_factors': N_FACTORS,
}
with open(f'{ARTIFACTS_DIR}/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_bundle, f)
print('✅ Saved svd_model.pkl')

## Step 6 — 列出所有 artifacts

In [ ]:
!ls -lh {ARTIFACTS_DIR}

## ✅ 完成

三個模型訓練 + 儲存完畢。下一步:打開 **03_evaluate.ipynb** 計算 Precision@10 / Recall@10 / NDCG@10。